# PHYS7101 · Exercise 1: The large-amplitude pendulum
Build a numerical model of the pendulum, check that you can trust it, use it to measure $g$ — and decide how far you trust your answer.

**Before you start:** File → Save a copy in Drive. Before submitting: Runtime → Restart session and run all.

Before each block of code, add a short Markdown cell saying what you are doing and why; after it, what the result means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit, brentq
from scipy.special import ellipk
import os

# Data: from a local datasets/ folder if you have the repository, otherwise straight from GitHub (e.g. in Colab).
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/coursework/exercise1/datasets/"
def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

d = np.loadtxt(data_path("pendulum_angle.csv"), delimiter=",", skiprows=1)
t = d[:, 0]              # time (s); t = 0 is the frame of release
theta_deg = d[:, 1]      # angle from the vertical (degrees)
theta = np.radians(theta_deg)

dt = 0.04                # frame interval (s): 25 frames per second
L = 0.800                # length (m), treat as exact
sigma = np.radians(0.5)  # tracking uncertainty per frame (rad), independent from frame to frame
print(f"{len(t)} frames, frame interval {dt:.4f} s, last frame at t = {t[-1]:.3f} s")

## Task 1 — Look at the data and write the model (Lecture 2)
Plot $\theta(t)$ for the whole record and for the first and last 10 s. Estimate the period; compare it with $2\pi\sqrt{L/g}$.
Write $\ddot\theta+\gamma\dot\theta+(g/L)\sin\theta=0$ as two first-order equations, and classify the system.

**Report:** the plots; your period estimate and $T_0$; the two first-order equations; the classification with reasons; the expected behaviour and the conserved quantity.

In [ ]:
# your code here

## Task 2 — Your own solver, and why you can trust it (Lectures 1 and 2)
2.1 energy test, undamped, $\theta_0=70^\circ$; 2.2 order of convergence; 2.3 choice of the step $h$. Write `f`, `euler` and `rk4` in the next cell.

**Checkpoint** (undamped, released from rest at $70^\circ$): after 10 s with $h = 0.04$ s, Euler's energy has grown about five-fold, while RK4's relative energy error is about $10^{-5}$. If yours differ by orders of magnitude, fix your solver before going on: everything after this uses it.

**Report:** the energy plot and your explanation; a table of $h$, errors and error ratios, and the order of each method; your $h$ and its $h/2$ difference compared with the uncertainty.

In [ ]:
def f(y, g, gamma):
    """Right-hand side of the pendulum equation: y = (theta, omega); return np.array([dtheta/dt, domega/dt])."""
    raise NotImplementedError

def euler(f, y0, t_end, h, *params):
    """Your Euler integrator for dy/dt = f(y, *params), with fixed step h.
    Return (times, y): times = 0, h, ..., N*h with N = round(t_end/h) (length N+1),
    and y of shape (N+1, 2) with columns theta and omega."""
    raise NotImplementedError

def rk4(f, y0, t_end, h, *params):
    """Your RK4 integrator, with the same arguments and the same return format as euler."""
    raise NotImplementedError

For Task 2.2: a high-accuracy reference at the same step times as your solvers.
`f` must have the same form as for your solvers: `f(y, *params)` returning an array `[dtheta/dt, domega/dt]`.

In [ ]:
def reference_solution(f, y0, t_end, h, *params):
    """High-accuracy solution at the times 0, h, 2h, ..., t_end (rows: times; columns: theta, omega)."""
    t_eval = np.arange(int(round(t_end/h)) + 1)*h
    sol = solve_ivp(lambda t, y: f(y, *params), (0, t_eval[-1]), y0, t_eval=t_eval,
                    method="DOP853", rtol=1e-12, atol=1e-12)
    return t_eval, sol.y.T

## Task 3 — Measure $g$ by fitting (Lecture 3)
3.1 the small-angle solution $\theta = A e^{-\beta t}\cos(\Omega t+\phi)$ with `curve_fit`; 3.2 your numerical model inside `curve_fit`; 3.3 residuals, $\chi^2/\nu$ (your own function), and comparison with a cited local value of $g$ at Lancaster.

Skeleton for 3.2: complete it with your own right-hand side `f` and your RK4 from Task 2. **Timing:** with $h = 0.01$ s, a fit to the whole record takes well under a minute. If it takes several minutes, check your solver (and test on the first 10 s first).

**Report:** $g$ and $\chi^2/\nu$ from 3.1; $\theta_0$, $g$ and $\gamma$ with their uncertainties, and $\chi^2/\nu$ (your own function), from 3.2; the figure and your explanation; the local $g$ with its source, and your difference from it in units of your uncertainty.

In [ ]:
h = None   # set this to your step from Task 2.3; the frame interval dt must be a whole number of steps

def model_theta(t_data, theta0, g, gamma):
    """theta (rad) at the times t_data (any frame times), for the pendulum released from rest at theta0.
    Works once f, rk4 and h are defined. Requires dt/h to be a whole number."""
    y0 = np.array([theta0, 0.0])                                   # theta(0) = theta0, omega(0) = 0
    times, y = rk4(f, y0, t_data.max(), h, g, gamma)               # your solver, from t = 0
    idx = np.rint(t_data/h).astype(int)                            # solver step at each requested time
    return y[idx, 0]

# Choose starting values from your Task 1 estimates, weight the fit by the frame uncertainty,
# and test it on the first 10 s before fitting the whole record.

In [ ]:
# your code here

## Task 4 — Period against amplitude (Lectures 3 and 4)
4.1 zero crossings with your own root finder on an interpolant; 4.2 period against amplitude;
4.3 exact period using `ellipk` (note: `ellipk(m)` takes $m=k^2$), with one value checked by your own Simpson or Gauss rule; 4.4 is the pendulum linear?

**Report:** the number of periods; the iterations per crossing and the largest difference from `brentq`; your check of $K$; the plot with both theoretical curves; your answers to 4.4.

In [ ]:
# your code here

## Task 5 — Angular velocity and extrapolation (Lecture 4)
5.1 $\omega$ by central differences, compared with the fitted model; 5.2 when does the amplitude fall below $1^\circ$?

**Report:** the rms difference between the two estimates of $\omega$, and the value you expect from the noise; the time at which the amplitude falls below 1°, and how far you trust it.

In [ ]:
# your code here

## Task 6 — Extension: what comes next?
Say why you chose your question; the reason should come from your results.

In [ ]:
# your code here

## Interpretive statement
(≤ 300 words, up to one figure; give the word count.)